# 12.7 General facilities for manipulating output

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

We describe here how some or all of AMPL's output can be directed to a file, and how
the volume of warning and error messages can be regulated.

**Redirection of output**

The examples in this book all show the outputs of commands as they would appear in
an interactive session, with typed commands and printed responses alternating. You may
direct all such output to a file instead, however, by adding a > and the name of the file:

```ampl
ampl: display ORIG, DEST, PROD >multi.out;
ampl: display supply >multi.out;
```

The first command that specifies >multi.out creates a new file by that name (or overwrites
any existing file of the same name). Subsequent commands add to the end of the
file, until the end of the session or a matching close command:

```ampl
ampl: close multi.out;
```

To open a file and append output to whatever is already there (rather than overwriting),
use >> instead of >. Once a file is open, subsequent uses of > and >> have the same
effect.

**Output logs**

The log_file option instructs AMPL to save subsequent commands and responses
to a file. The option's value is a string that is interpreted as a filename:

```ampl
ampl: option log_file 'multi.tmp';
```

The log file collects all AMPL statements and the output that they produce, with a few
exceptions described below. Setting log_file to the empty string:

```ampl
ampl: option log_file ";
```

turns off writing to the file; the empty string is the default value for this option.

When AMPL reads from an input file by means of a `model` or `data` command (or an
include command as defined in [Chapter 13](#missing) <!--- xTODO missing reference --->), the statements from that file are not normally
copied to the log file. To request that AMPL echo the contents of input files,
change option log_model (for input in `model` mode) or log_data (for input in `data`
mode) from the default value of 0 to a nonzero value.

When you invoke a solver, AMPL logs at least a few lines summarizing the `objective`
value, solution status and work required. Through solver-specific directives, you can typically
request additional solver output such as logs of iterations or branch-and-bound
nodes. Many solvers automatically send all of their output to AMPL's log file, but this
compatibility is not universal. If a solver's output does not appear in your log file, you
should consult the supplementary documentation for that solver's AMPL interface; possibly
that solver accepts nonstandard directives for diverting its output to files.

**Limits on messages**

By specifying option eexit n, where n is some `integer`, you determine how AMPL
handles error messages. If n is not zero, any AMPL statement is terminated after it has
produced abs(n) error messages; a negative value causes only the one statement to be
terminated, while a positive value results in termination of the entire AMPL session. The
effect of this option is most often seen in the use of `model` and `data` statements where
something has gone badly awry, like using the wrong file:

```ampl
ampl: option eexit -3;
ampl: model diet.mod;
ampl: data diet.mod;
diet.mod, line 4 (offset 32):
       expected ; ( [ : or symbol
context: param cost >>> { <<< FOOD} > 0;
diet.mod, line 5 (offset 56):
       expected ; ( [ : or symbol
context: param f_min >>> { <<< FOOD} >= 0;
diet.mod, line 6 (offset 81):
       expected ; ( [ : or symbol
context: param f_max >>> { <<< j in FOOD} >= f_min[j];
Bailing out after 3 warnings.
```

The default value for eexit is -10. Setting it to 0 causes all error messages to be displayed.

The eexit setting also applies to infeasibility warnings produced by AMPL's presolve
phase after you type `solve`. The number of these warnings is simultaneously limited
by the value of option presolve_warnings, which is typically set to a smaller
value; the default is 5.
An AMPL `data` statement may specify values that correspond to illegal combinations
of indices, due to any number of mistakes such as incorrect index sets in the `model`,
indices in the wrong order, misuse of (tr), and typing errors. Similar errors may be
caused by `let` statements that change the membership of index sets. AMPL catches these
errors after `solve` is typed. The number of invalid combinations displayed is limited to
the value of the option bad_subscripts, whose default value is 3.